# CANopen — SDO, PDO, NMT, dan heartbeat

Master dan modul I/O simulasi bergaya CiA 401 berbagi bus CAN virtual. Ganti bus dengan `await CanBus.OpenAsync("socketcan:can0")` (atau `slcan:`, `gsusb:`, `pcan:usb1`) untuk menjangkau node sungguhan.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.18.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.18.0-preview.1"

## Nyalakan node dan pindai jaringan
Modul mengirim pesan boot-up, lalu heartbeat; pemindaian membaca 0x1000, 0x1008, dan 0x1018 lewat SDO.

In [ ]:
using IoTCom.Net.Protocols.CanOpen;
using IoTCom.Net.Transport.Can;

var coNet = new VirtualCanNetwork("notebook");
var ioModule = CanOpenIoModuleSimulator.Create(coNet.CreateNode(), nodeId: 5, heartbeatMs: 200, eventTimerMs: 150);
var coBus = coNet.CreateNode();
var coMaster = CanOpenMaster.Create(coBus);
await coMaster.StartAsync();
await ioModule.StartAsync();
foreach (var n in await coMaster.ScanAsync(1, 10, TimeSpan.FromMilliseconds(50)))
    Console.WriteLine($"node {n.Id}: {n.Name}, type 0x{n.DeviceType:X8}, vendor 0x{n.Identity?.Vendor:X8}, serial {n.Identity?.Serial:X8}, {n.State}");

## SDO: expedited, bersegmen, dan abort
Nilai hingga 4 byte muat dalam satu frame; yang lebih panjang dipecah menjadi segmen 7 byte dengan bit toggle.

In [ ]:
Console.WriteLine(await coMaster.ReadAsync(5, 0x6401, 1, CanOpenDataType.Integer16));             // expedited
Console.WriteLine(await coMaster.ReadAsync(5, 0x2100, 0, CanOpenDataType.VisibleString));          // segmented
await coMaster.WriteAsync(5, 0x2100, 0, CanOpenDataType.VisibleString, "Pump skid 9, Bekasi");
Console.WriteLine(ioModule.Node.Dictionary[0x2100, 0]);
try { await coMaster.UploadAsync(5, 0x9999, 0); } catch (CanOpenSdoException e) { Console.WriteLine(e.Message); }
try { await coMaster.WriteAsync(5, 0x1000, 0, CanOpenDataType.Unsigned32, 1); } catch (CanOpenSdoException e) { Console.WriteLine(e.Message); }

## NMT start, PDO, dan SYNC
Hanya node operational yang mengirim PDO. TPDO1 digerakkan event dengan event timer; TPDO2 menjawab setiap SYNC.

In [ ]:
var mapping = await coMaster.ReadTpdoMappingAsync(5, 1);
Console.WriteLine(string.Join(", ", mapping.Objects));
var pdoCount = 0;
coMaster.PdoReceived += (node, pdo, data) => { if (pdoCount++ < 4) Console.WriteLine($"TPDO{pdo} node {node}: {Convert.ToHexString(data)}"); };
await coMaster.NmtAsync(NmtCommand.Start, 5);
await coMaster.WriteAsync(5, 0x6200, 1, CanOpenDataType.Unsigned8, 1);                             // pump on
ioModule.Step();
await coMaster.SyncAsync();
await Task.Delay(400);
Console.WriteLine($"flow {ioModule.Node.Dictionary[0x6401, 2].Value} (0.1 l/min), state {coMaster.Nodes.First().State}");

## Di jalur

In [ ]:
foreach (var f in CanOpenCodec.Describe(new CanFrame(0x585, Convert.FromHexString("410810000B000000"))))
    Console.WriteLine($"{f.Name,-8} {f.Value}");
await ioModule.DisposeAsync();
await coMaster.DisposeAsync();

## Lebih lanjut
`iotcom canopen scan --can sim`, `iotcom canopen monitor --can sim`, *Modul I/O CANopen* di Gallery, dan sampel CanOpenMaster. Lihat `docs/id/protocols/canopen.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*